# 01.04 · Probability, softmax & entropy

**Chapter:** Python & Math for AI · **Milestone:** M1

## Goals
- [ ] Distributions, expectation, variance; Bayes' rule
- [ ] Softmax and temperature: how an LLM turns scores into a next-token distribution
- [ ] Entropy, cross-entropy, perplexity: the loss every LLM is trained on

## Resources
- [Seeing Theory (visual probability)](https://seeing-theory.brown.edu/) — free
- [Khan Academy – Statistics & probability](https://www.khanacademy.org/math/statistics-probability) — free
- [Chris Olah – Visual Information Theory](https://colah.github.io/posts/2015-09-Visual-Information/) — free

An LLM is a **probability distribution over the next token**. Everything about sampling, temperature, logprobs,
training loss and confidence follows from a handful of ideas below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## 1. Distributions, expectation, variance
A discrete distribution assigns probabilities to outcomes (summing to 1). **Expectation** is the probability-weighted
average; **variance** is the expected squared distance from it. Sample averages converge to the expectation (law of large numbers).

In [ ]:
outcomes = np.arange(1, 7)
p = np.full(6, 1 / 6)  # fair die
E = (outcomes * p).sum()
Var = (((outcomes - E) ** 2) * p).sum()
samples = rng.choice(outcomes, size=10_000, p=p)
print(f"E={E:.3f} Var={Var:.3f} | sample mean={samples.mean():.3f} sample var={samples.var():.3f}")

running_mean = np.cumsum(samples) / np.arange(1, len(samples) + 1)
plt.plot(running_mean); plt.axhline(E, color="r", ls="--"); plt.xscale("log")
plt.title("Law of large numbers: running mean of die rolls"); plt.show()

## 2. Conditional probability & Bayes
$P(A|B) = \frac{P(B|A)P(A)}{P(B)}$. Classic trap: a 99%-accurate detector on a rare event is mostly wrong when it fires.
You'll meet this again when evaluating LLM guardrails and judges (chapters 10-11): base rates matter.

In [ ]:
p_injection = 0.01  # 1% of prompts are prompt-injection attacks
p_flag_given_inj = 0.99  # detector recall
p_flag_given_ok = 0.05  # detector false-positive rate
p_flag = p_flag_given_inj * p_injection + p_flag_given_ok * (1 - p_injection)
print(f"P(injection | flagged) = {p_flag_given_inj * p_injection / p_flag:.1%}")

## 3. Softmax and temperature
A model outputs one raw score (**logit**) per vocabulary token. Softmax turns logits into probabilities:
$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$.
**Temperature** $T$ divides logits first: $T<1$ sharpens (more deterministic), $T>1$ flattens (more random), $T \to 0$ = argmax (greedy).

In [ ]:
def softmax(z, T=1.0):
    z = np.asarray(z, dtype=float) / T
    z = z - z.max()  # subtract max for numerical stability (doesn't change the result)
    e = np.exp(z)
    return e / e.sum()

vocab = ["Paris", "London", "Rome", "Berlin", "pizza"]
logits = np.array([5.0, 3.5, 3.0, 2.5, 0.5])  # "The capital of France is ..."

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, T in zip(axes, [0.3, 1.0, 2.0]):
    ax.bar(vocab, softmax(logits, T)); ax.set_title(f"T = {T}")
plt.tight_layout(); plt.show()

print("overflow without max-subtraction:", np.exp(np.array([1000.0])), "| stable softmax:", softmax([1000.0, 999.0]))

## 4. Sampling
Generating text = repeatedly sampling a token from this distribution. `rng.choice(vocab, p=probs)` is literally what happens
(plus tricks like top-p, chapter 04.06).

In [ ]:
probs = softmax(logits, T=1.0)
draws = rng.choice(vocab, size=1000, p=probs)
values, counts = np.unique(draws, return_counts=True)
print(dict(zip(values, counts)))

## 5. Entropy, cross-entropy, perplexity
- **Entropy** $H(p) = -\sum p_i \log p_i$: how uncertain a distribution is (0 = certain).
- **Cross-entropy** $H(p, q) = -\sum p_i \log q_i$: how surprised model $q$ is by the true distribution $p$.
  With a one-hot target (the actual next token), it's just $-\log q_{\text{correct}}$. **This is the LLM training loss.**
- **Perplexity** $= e^{\text{mean cross-entropy}}$: "the model is as confused as choosing among N tokens uniformly".
- **KL divergence** $= H(p,q) - H(p)$: distance between distributions; used in RLHF/DPO to keep a model near its reference.

In [ ]:
def entropy(p):
    p = np.asarray(p); p = p[p > 0]
    return -(p * np.log(p)).sum()

print("entropy uniform(5):", round(entropy(np.full(5, 0.2)), 3), "= log 5 =", round(np.log(5), 3))
print("entropy at T=0.3 / 1 / 2:", [round(entropy(softmax(logits, T)), 3) for T in (0.3, 1, 2)])

# cross-entropy loss over a short "sequence" where the correct token index is known at each step
model_probs = np.array([softmax(logits) for _ in range(4)])  # (seq, vocab)
targets = np.array([0, 0, 1, 4])  # Paris, Paris, London, pizza
token_losses = -np.log(model_probs[np.arange(4), targets])
print("per-token loss:", token_losses.round(2), "| mean:", token_losses.mean().round(3), "| perplexity:", np.exp(token_losses.mean()).round(2))

Note how the "pizza" token costs ~4.6 nats: the model assigned it ~1% probability. Training pushes down the loss on
whatever token actually came next in the data.

## Exercises
1. Sample 1000 tokens at T = 0.2, 1, 2 and plot the three histograms side by side. What happens to "pizza"?
2. What is the cross-entropy of a model that's uniform over a 50,000-token vocabulary? Its perplexity?
   (Compare: a decent small LLM has perplexity ~10-20 on web text.)
3. Implement `kl(p, q)` and show it's ≥ 0 and not symmetric.

In [ ]:
# your code here